<a href="https://colab.research.google.com/github/yukisirosui/AI-Driven-Python-Study-Tracker-with-GitHub-Integration/blob/main/GitHub%E5%90%8C%E6%9C%9F_AI_Python%E3%82%B3%E3%83%BC%E3%83%87%E3%82%A3%E3%83%B3%E3%82%B0%E5%AD%A6%E7%BF%92%E3%83%A1%E3%83%B3%E3%82%BF%E3%83%BC%E3%82%B7%E3%82%B9%E3%83%86%E3%83%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🤖 Python学習習慣化システム (AIメンター・フォーム確定版) { display-mode: "form" }
#@markdown 最初にこのセルを実行して、システムを起動させてください。

import os
import json
import datetime
import requests
import base64
from google.colab import userdata
import google.generativeai as genai

try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel('models/gemini-2.5-flash')
    print("✅ システムの起動に成功しました！下の各コントロールフォームへ進んでください。")
except Exception as e:
    print("【エラー】ColabのSecretsに 'GEMINI_API_KEY' が設定されているか確認してください。")

LOG_FILE = "python_study_log.json"

def _internal_generate(level):
    history_summary = "なし"
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            try:
                logs = json.load(f)
                past_topics = []
                for day in logs:
                    for ch in day.get("challenges", []):
                        past_topics.append(ch.get("topic", ""))
                history_summary = ", ".join(past_topics[-5:])
            except: pass

    prompt = f"""
    AIプログラミングメンターとして、学習者の条件（レベル: {level}、最近解いたテーマ: {history_summary}）に合う課題を1問作成してください。
    【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
    {{
        "title": "課題のタイトル",
        "topic": "学習テーマ",
        "description": "問題の具体的な説明",
        "example_input": "入力例",
        "example_output": "期待される出力例"
    }}
    """
    response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
    return json.loads(response.text)

def _internal_hint(challenge):
    print("🤔 メンターがヒントを考えています...")
    prompt = f"""
    学習者が以下の課題で行き詰まっています。解答コードそのものは絶対に明かさず、
    実装の手がかりとなるヒントを3段階で優しく教えてください。
    課題: {challenge['title']}\n内容: {challenge['description']}
    """
    response = model.generate_content(prompt)
    print(f"\n💡 【メンターからのヒント】\n{response.text}")

def _internal_review(challenge, user_code):
    print("\n🩺 AIメディックがコードを診断中...")
    prompt = f"""
    出題した課題に対して、学習者が書いたコードをレビューし、採点とアドバイスをしてください。
    【課題】{challenge['title']} / {challenge['description']}
    【コード】\n```python\n{user_code}\n```
    【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
    {{
        "status": "「クリア」または「要修正」",
        "score": 100点満点中の点数(数値),
        "review": "コードのアドバイス",
        "refactored_code": "60点以上の場合は見本コード。60点未満の場合は空欄"
    }}
    """
    response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
    result = json.loads(response.text)

    print(f"\n📊 採点結果: {result['score']}点 【{result['status']}】")
    print(f"💬 メンターのアドバイス:\n{result['review']}")

    if result['score'] < 60:
        print("\n⚠️ 【判定: 不合格】点数が60点に満たなかったため、今回のコードは記録されません。")
        return

    if result['refactored_code']:
        print(f"\n💡 模範コード（リファクタリング案）:\n{result['refactored_code']}")

    _internal_save_and_push(challenge, user_code, result['score'])

def _internal_save_and_push(challenge, user_code, score):
    today_str = str(datetime.date.today())
    logs = []
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            try: logs = json.load(f)
            except: pass

    challenge_data = {
        "time": datetime.datetime.now().strftime("%H:%M:%S"),
        "title": challenge['title'],
        "topic": challenge['topic'],
        "problem_description": challenge['description'],
        "my_code": user_code,
        "score": score
    }

    existing_day_index = -1
    for i, log in enumerate(logs):
        if log.get("date") == today_str:
            existing_day_index = i
            break

    if existing_day_index != -1:
        if not isinstance(logs[existing_day_index], dict) or "challenges" not in logs[existing_day_index]:
            logs[existing_day_index] = {"date": today_str, "total_solved_today": 0, "challenges": []}
        logs[existing_day_index]["challenges"].append(challenge_data)
        logs[existing_day_index]["total_solved_today"] = len(logs[existing_day_index]["challenges"])
        print(f"\n🔄 本日（{today_str}）の{logs[existing_day_index]['total_solved_today']}問目の合格コードを履歴に追加しました！")
        is_new_day = False
    else:
        new_day_data = {
            "date": today_str,
            "total_solved_today": 1,
            "challenges": [challenge_data]
        }
        logs.append(new_day_data)
        print(f"\n🔥 今日の最初の学習スタンプを記録しました！ (累計クリア日数: {len(logs)}日)")
        is_new_day = True

    content_str = json.dumps(logs, ensure_ascii=False, indent=4)
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write(content_str)

    try:
        GITHUB_USER = "yukisirosui"
        REPO_NAME = "python-daily-study"  # ⭕ あなたの実際のGitHubリポジトリ名に修正
        token = userdata.get('GITHUB_TOKEN')

        # ⭕ APIのURLには「repos」が必要です
        url = f"https://api.github.com/repos/{GITHUB_USER}/{REPO_NAME}/contents/{LOG_FILE}"

        headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}

        res = requests.get(url, headers=headers)
        sha = res.json().get("sha") if res.status_code == 200 else None

        commit_msg = f"🌱 Add New Day: {today_str}" if is_new_day else f"🚀 Add Another Solved Challenge: {today_str}"
        data = {
            "message": f"🤖 {commit_msg} ({challenge['title']})",
            "content": base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
        }
        if sha: data["sha"] = sha
        put_res = requests.put(url, headers=headers, json=data)

        # 修正：成功ステータス（200番台）を正しく判定
        if put_res.status_code in [200, 201]:
            print(f"✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱")
        else:
            print(f"⚠️ GitHub同期エラー: {put_res.json().get('message')}")
    except Exception as e:
        print(f"❌ GitHub連携中にエラーが発生しました: {e}")

✅ システムの起動に成功しました！下の各コントロールフォームへ進んでください。


In [ ]:
#@title 📅 ステップ1: 課題の生成＆ヒントフォーム { display-mode: "form" }
現在のレベル = "初級" #@param ["初級", "中級", "上級"]
ヒントを表示する = False #@param {type:"boolean"}

import datetime # datetimeモジュールをインポート

# チャレンジを生成または既存のものを使用
if not ヒントを表示する: # ヒントを表示しない場合、常に新しい課題を生成
    today_challenge = _internal_generate(現在のレベル)
elif 'today_challenge' not in locals(): # ヒントを表示する場合で、まだ課題が生成されていない場合
    print("⚠️ 課題がまだ生成されていません。新しい課題を生成します。")
    today_challenge = _internal_generate(現在のレベル)
# else: ヒントを表示する場合で、すでに課題が生成されている場合は既存のものを使用

# 常に問題文を表示
print(f"📅 【本日（{datetime.date.today()}）の課題】")
print(f"📌 タイトル: {today_challenge['title']} (テーマ: {today_challenge['topic']})")
print(f"📝 問題内容:\n{today_challenge['description']}")
if 'example_input' in today_challenge and 'example_output' in today_challenge:
    print(f"📥 入力例: {today_challenge['example_input']}\n📤 出力例: {today_challenge['example_output']}")
print("--------------------------------------------------")

# ヒントを表示するがTrueの場合のみヒントを表示
if ヒントを表示する:
    _internal_hint(today_challenge)
    print("--------------------------------------------------")
else:
    print("💡 ヒントが必要な場合は `ヒントを表示する` をTrueにして再度このセルを実行してください。")

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1880.21ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1953.33ms


📅 【本日（2026-05-25）の課題】
📌 タイトル: 2つの整数の合計計算 (テーマ: 基本的な入出力と算術演算)
📝 問題内容:
ユーザーから2つの整数を受け取り、それらの合計を計算して出力するプログラムを作成してください。
📥 入力例: 10
20
📤 出力例: 30
--------------------------------------------------
💡 ヒントが必要な場合は `ヒントを表示する` をTrueにして再度このセルを実行してください。


In [ ]:
#@title 💻 ステップ2: 解答提出フォーム { display-mode: "form" }
#@markdown ---
#@markdown ### 📝 あなたのPythonコードをここに記述してください
#@markdown 上記の課題に対する解答コードを、以下のトリプルクォーテーション `"""` の間に直接入力してください。
#@markdown コードを記述し終えたら、このセルを実行してAIメンターからの診断を受けましょう。
#@markdown ---
あなたの回答コード = """
    x = input("ここに整数を記入してください：")
    y = input("ここに整数を記入してください：")
    total = int(x) + int(y)
    print(f"{x} + {y} = {total}")
"""

if 'today_challenge' in locals():
    _internal_review(today_challenge, あなたの回答コード)
else:
    print("⚠️ 先に『ステップ1の課題生成』を実行してください。")


🩺 AIメディックがコードを診断中...

📊 採点結果: 95点 【クリア】
💬 メンターのアドバイス:
### 良い点
*   `input()`関数を使ってユーザーからの入力を正しく受け取っています。
*   `int()`関数を使って、受け取った文字列を適切に整数に変換できています。
*   f-stringを使うことで、計算結果を分かりやすく、かつ整形して出力できています。
*   課題の要件（2つの整数を受け取り、合計を計算して出力する）をすべて満たしています。

### 改善点とアドバイス
*   **エラーハンドリングの追加:** 現在のコードでは、ユーザーが「abc」のような整数以外の値を入力した場合、`int()`関数が`ValueError`を発生させ、プログラムがクラッシュしてしまいます。より堅牢なプログラムにするためには、`try-except`ブロックを使用してこのようなエラーを捕捉し、ユーザーに適切なメッセージを表示したり、再入力を促したりする仕組みを追加すると良いでしょう。
*   **入力プロンプトの具体性:** 「ここに整数を記入してください：」というプロンプトは機能しますが、「最初の整数を入力してください:」や「2番目の整数を入力してください:」のように、もう少し具体的にすると、ユーザーにとってより分かりやすくなります。

💡 模範コード（リファクタリング案）:
first_num_str = input("最初の整数を入力してください: ")
second_num_str = input("2番目の整数を入力してください: ")

try:
    num1 = int(first_num_str)
    num2 = int(second_num_str)
    total = num1 + num2
    print(f"{num1} + {num2} = {total}")
except ValueError:
    print("エラー: 有効な整数が入力されませんでした。プログラムを再実行して、整数を入力してください。")

🔄 本日（2026-05-25）の2問目の合格コードを履歴に追加しました！
✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱


In [ ]:
#@title 🔍 オプション: AI辞書フォーム { display-mode: "form" }
調べたい単語 = "`try-except`" #@param {type:"string"}

prompt = f"""
あなたは親切なPythonプログラミング講師です。「{調べたい単語}」について解説してください。
1. 【一言でいうと】: 簡単な説明
2. 【詳しい仕組み】: Pythonにおける役割
3. 【具体的なコード例】: 短いコードと実行結果
"""
print(f"🔍 『{調べたい単語}』についてAIメンターが調べています...")
response = model.generate_content(prompt)
print("\n" + "="*50 + f"\n📖 【AI辞書】単語解説: {調べたい単語}\n" + "="*50)
print(response.text)


🔍 『`try-except`』についてAIメンターが調べています...

📖 【AI辞書】単語解説: `try-except`
こんにちは！Pythonプログラミング講師の〇〇です。今日はプログラムをより頑丈にするためのとっても大切な仕組み、「`try-except`」について、一緒に楽しく学んでいきましょうね！

---

### 1. 【一言でいうと】

`try-except`は、**プログラムがエラーで突然止まっちゃうのを防いで、「もしもエラーが起きたら、こう対処してね」と指示するための仕組み**、と覚えてください！ まさに「転ばぬ先の杖」ですね！

### 2. 【詳しい仕組み】

Pythonのプログラムは、通常、上から順番に実行されていきます。でも、時々予期せぬ事態が起こることがあります。例えば、ユーザーが数値を入力するはずの場所で、間違って文字を入力してしまったり、存在しないファイルを読み込もうとしたり…。

こんな時、プログラムは「あれ？想定外だ！」とパニックになって、そこで実行を中断しちゃいます。これを「例外（Exception）が発生した」と言います。プログラムが途中で止まってしまうと、ユーザーにとっては使いづらく、時には困ったことになってしまいますよね。

そこで`try-except`の出番です！

*   **`try`ブロック**:
    *   ここに「**もしかしたらエラーが起こるかもしれない、ちょっと危険な処理**」を書きます。
    *   Pythonはまずこの`try`ブロックの中のコードを実行しようとします。

*   **`except`ブロック**:
    *   もし`try`ブロックの中でエラー（例外）が発生した場合、Pythonはすぐに`try`ブロックの残りの処理を中断して、**`except`ブロックに書かれた処理を実行しに行きます**。
    *   エラーの種類を指定することもできます（例: `except ValueError:`）。そうすると、指定した種類のエラーだけをキャッチして処理できます。

このように、`try-except`を使うことで、プログラムが途中で停止してしまうことを防ぎ、エラーが起きた際も落ち着いて別の処理を実行したり、ユーザーに分かりやすいメッセージを表示したりすること